In [15]:
!kaggle datasets download -d emmarex/plantdisease

Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
plantdisease.zip: Skipping, found more recently modified local copy (use --force to force download)


In [16]:
import zipfile

with zipfile.ZipFile('plantdisease.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')

In [17]:
import os

print(os.listdir('/content/data'))

['PlantVillage', 'plantvillage', 'train', 'valid']


In [18]:
from sklearn.model_selection import train_test_split
import os, shutil

src = '/content/data/PlantVillage'   # folder gốc
train_dir = '/content/data/train'
test_dir  = '/content/data/valid'

for cls in os.listdir(src):
    cls_path = os.path.join(src, cls)

    if not os.path.isdir(cls_path):
        continue

    files = os.listdir(cls_path)

    train_files, test_files = train_test_split(
    files, test_size=0.2, random_state=42, shuffle=True
)

    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

    for f in train_files:
        shutil.copy(os.path.join(cls_path, f),
                    os.path.join(train_dir, cls, f))

    for f in test_files:
        shutil.copy(os.path.join(cls_path, f),
                    os.path.join(test_dir, cls, f))

In [19]:
print(os.listdir('/content/data'))

['PlantVillage', 'plantvillage', 'train', 'valid']


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "gpu")
print(device)

cuda


In [22]:
transform_train = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

transform_test = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [23]:
train_path = '/content/data/train'
test_path  = '/content/data/valid'

train_dataset = datasets.ImageFolder(train_path, transform=transform_train)
test_dataset  = datasets.ImageFolder(test_path, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32)

In [24]:
num_classes = len(train_dataset.classes)
print(num_classes)

15


In [25]:
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(256*8*8, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))

        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [26]:
model = CNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

In [27]:
num_epochs = 15

train_acc_list = []
train_loss_list = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    train_loss_list.append(epoch_loss)
    train_acc_list.append(epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} |  Acc: {epoch_acc:.2f}%")

Epoch 1/15 | Loss: 1.2619 |  Acc: 60.91%
Epoch 2/15 | Loss: 0.7235 |  Acc: 75.76%
Epoch 3/15 | Loss: 0.5602 |  Acc: 81.56%
Epoch 4/15 | Loss: 0.5035 |  Acc: 83.49%
Epoch 5/15 | Loss: 0.4192 |  Acc: 85.74%
Epoch 6/15 | Loss: 0.3781 |  Acc: 87.78%
Epoch 7/15 | Loss: 0.3600 |  Acc: 88.23%
Epoch 8/15 | Loss: 0.3180 |  Acc: 89.51%
Epoch 9/15 | Loss: 0.2920 |  Acc: 90.01%
Epoch 10/15 | Loss: 0.2872 |  Acc: 90.51%
Epoch 11/15 | Loss: 0.2626 |  Acc: 91.34%
Epoch 12/15 | Loss: 0.2466 |  Acc: 91.72%
Epoch 13/15 | Loss: 0.2402 |  Acc: 92.14%
Epoch 14/15 | Loss: 0.2298 |  Acc: 92.30%
Epoch 15/15 | Loss: 0.2087 |  Acc: 92.88%


In [29]:
torch.save(model.state_dict(), "model_Plant.pth")

In [28]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print("Test Accuracy:", test_accuracy)

Test Accuracy: 96.1780358006773
